In [ ]:
from herbie import Herbie
from herbie import paint
from herbie.toolbox import EasyMap, pc, ccrs
import xarray as xr
import rioxarray as rxr

import matplotlib.pyplot as plt

In [ ]:
H = Herbie("2023-08-01", product="sfc")
ds = H.xarray("(?:HGT|LAND):surface")
ds

In [ ]:
# We want to set the water points to nan
ds["orog"] = ds.orog.where(ds.lsm > 0)

In [ ]:
ax = EasyMap("50m", figsize=[15, 9], crs=ds.herbie.crs).STATES().ax
p = ax.pcolormesh(
    ds.longitude,
    ds.latitude,
    ds.orog,
    transform=pc,
    cmap=paint.LandGreen.cmap,
)

plt.colorbar(
    p,
    ax=ax,
    orientation="horizontal",
    pad=0.01,
    shrink=0.8,
    label="Model Terrain Height (m)",
)

In [ ]:
preds = xr.open_dataset("conus_preds.nc")

In [ ]:
preds.sel(band="preds").isel(time=0).band_data

In [ ]:
ds['rnn_preds'] = preds.sel(band="preds").isel(time=0).band_data

In [ ]:
ax = EasyMap("50m", figsize=[15, 9], crs=ds.herbie.crs).STATES().ax
p = ax.pcolormesh(
    ds.longitude,
    ds.latitude,
    ds.rnn_preds,
    transform=pc,
    cmap=paint.NWSRelativeHumidity.cmap,
)

plt.colorbar(
    p,
    ax=ax,
    orientation="horizontal",
    pad=0.01,
    shrink=0.8,
    label="Fuel Moisture Content (%)",
)

In [ ]:
ds_spinfo = rxr.open_rasterio("20240420/hrrr.t00z.wrfprsf02.629.tif")

In [ ]:
import sys
sys.path.append('..')
from moisture_rnn_xarray import bbox_to_xy

bbox = [37, -111, 46, -95]         # Spatial bounding box
minx, miny, maxx, maxy = bbox_to_xy(bbox, ds.herbie.crs)

In [ ]:
ds['rnn_preds'] = preds.sel(band="preds").isel(time=0).band_data

# Create a mask for elements within the bounding box
# mask = (ds.y >= miny) & (ds.y <= maxy) & (ds.x >= minx) & (ds.x <= maxx)
# ds['rnn_preds'] = ds['rnn_preds'].where(mask)

In [ ]:

ax = EasyMap("50m", figsize=[15, 9], crs=ds.herbie.crs).STATES().ax
p = ax.pcolormesh(
    ds.longitude,
    ds.latitude,
    ds.rnn_preds,
    transform=pc,
    cmap=paint.NWSRelativeHumidity.cmap,
)

plt.colorbar(
    p,
    ax=ax,
    orientation="horizontal",
    pad=0.01,
    shrink=0.8,
    label="Fuel Moisture Content (%)",
)

In [ ]:
# Get the CRS from the dataset
dataset_crs = ds.herbie.crs

# Set up a transformer to go from WGS84 to the dataset CRS
transformer_to_crs = Transformer.from_crs("EPSG:4326", dataset_crs, always_xy=True)

# Set up a transformer to go from the dataset CRS back to WGS84
transformer_to_lonlat = Transformer.from_crs(dataset_crs, "EPSG:4326", always_xy=True)

# Convert the bbox into projected coordinates (dataset CRS)
min_x, min_y = transformer_to_crs.transform(bbox[1], bbox[0])  # min_lon, min_lat
max_x, max_y = transformer_to_crs.transform(bbox[3], bbox[2])  # max_lon, max_lat

# Convert back to geographic coordinates (longitude and latitude)
min_lon, min_lat = transformer_to_lonlat.transform(min_x, min_y)
max_lon, max_lat = transformer_to_lonlat.transform(max_x, max_y)

In [ ]:
min_lon

In [ ]:
min_x

In [ ]:
adjusted_bbox = [
    bbox[0],                            # min_lat
    bbox[1] + 360 if bbox[1] < 0 else bbox[1],  # min_lon adjusted
    bbox[2],                            # max_lat
    bbox[3] + 360 if bbox[3] < 0 else bbox[3],  # max_lon adjusted
]

In [ ]:
lat = ds.latitude
lon = ds.longitude

ds['rnn_preds'] = preds.sel(band="preds").isel(time=0).band_data

mask = (
    (lat >= adjusted_bbox[0]) & (lat <= adjusted_bbox[2]) &  # Latitude range
    (lon >= adjusted_bbox[1]) & (lon <= adjusted_bbox[3])   # Longitude range
)


ds['rnn_preds'] = ds['rnn_preds'].where(mask)

In [ ]:
ax = EasyMap("50m", figsize=[15, 9], crs=ds.herbie.crs).STATES().ax
p = ax.pcolormesh(
    ds.longitude,
    ds.latitude,
    ds.rnn_preds,
    transform=pc,
    cmap=paint.NWSRelativeHumidity.cmap,
)

plt.colorbar(
    p,
    ax=ax,
    orientation="horizontal",
    pad=0.01,
    shrink=0.8,
    label="Fuel Moisture Content (%)",
)